## 1. Импорт зависимостей

In [41]:
import sys
from pathlib import Path
import polars as pl
import pandas as pd
import json
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from match import CONFIG_DIR, resolve_project_path

with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="prepare_data")

pl.Config.set_tbl_rows(-1)       # показывать все строки
pl.Config.set_tbl_cols(-1)       # показывать все столбцы
pl.Config.set_fmt_str_lengths(1000)  # не обрезать длинные строки
pl.Config.set_tbl_width_chars(200)   # ширина таблицы

polars.config.Config

## 2. Чтение конфига

In [27]:
cfg

{'path': {'items_human_path': 'data/items_human_normalized.parquet', 'matches_human_path': 'data/matches.parquet', 'items_human_normalized_path': 'data/items_human_normalized.parquet'}}

## 3. Чтение датасета

In [39]:
project = Path.cwd().parent
items_human_path = project / Path(cfg["path"]["items_human_path"])
matches_human_path = project / Path(cfg["path"]["matches_human_path"])
df_human = pl.read_parquet(items_human_path)
df_matches = pl.read_parquet(matches_human_path)
df_human, df_matches

(shape: (711_304, 5)
 ┌──────────────┬──────────────────────────────────────────────────────┬──────────────────────────────────────────────────────┬───────────────────┬─────────────────────────────────────────────────────┐
 │ id           ┆ name                                                 ┆ attributes                                           ┆ category          ┆ normalized_attributes                               │
 │ ---          ┆ ---                                                  ┆ ---                                                  ┆ ---               ┆ ---                                                 │
 │ i64          ┆ str                                                  ┆ str                                                  ┆ str               ┆ str                                                 │
 ╞══════════════╪══════════════════════════════════════════════════════╪══════════════════════════════════════════════════════╪═══════════════════╪════════════════════════

## 3.2 Разбор карточек которые совпадают

In [29]:
df_matches.filter(pl.col("target") == 1).head()

id1,id2,target
i64,i64,f64
476,841813632218,1.0
525,13777,1.0
622,360777364649,1.0
706,352187426187,1.0
753,747324374388,1.0


In [125]:
row = df_matches.filter(pl.col("target") == 1).sample(1)
id1, id2 = row.select("id1", "id2").row(0)


item1 = (
    df_human
    .filter(pl.col("id") == id1)
    .select("id", "name", "normalized_attributes")
    .row(0, named=True)
)

item2 = (
    df_human
    .filter(pl.col("id") == id2)
    .select("id", "name", "normalized_attributes")
    .row(0, named=True)
)


attrs1 = json.loads(item1["normalized_attributes"])
attrs2 = json.loads(item2["normalized_attributes"])

all_keys = sorted(set(attrs1) | set(attrs2))


names = pl.DataFrame({
    "id": [item1["id"], item2["id"]],
    "name": [item1["name"], item2["name"]],
})

comparison = (
    pl.DataFrame({
        "attribute": all_keys,
        f"value_{id1}": [attrs1.get(key) for key in all_keys],
        f"value_{id2}": [attrs2.get(key) for key in all_keys],
    })
    .with_columns(
        (
            pl.col(f"value_{id1}").fill_null("<нет>")
            == pl.col(f"value_{id2}").fill_null("<нет>")
        ).alias("equal")
    )
    .sort(["equal", "attribute"], descending=[True, False])
)

display(names)
display(comparison)

id,name
i64,str
575525714050,"""lori / набор для творчества lori картина из пай"""
326417531997,"""набор для творчества картина из пайеток единорог ап-047"""


attribute,value_575525714050,value_326417531997,equal
str,str,str,bool
"""бренд""",null,"""нет бренда""",false
"""валюта""","""rub""",null,false
"""высота упаковки, мм""","""40""",null,false
"""длина упаковки, мм""","""260""",null,false
"""комплект""","""наборы для поделок - 1 шт""",null,false
"""страна-производитель""",null,"""россия""",false
"""тип""",null,"""картина из пайеток""",false
"""ширина упаковки, мм""","""220""",null,false


## 4. Дисбаланс класса 1/4

In [31]:
attrs = json.loads(df_human.filter(pl.col("id") == id1).select("attributes").to_series().to_list()[0])
attrs.keys()

dict_keys(['бренд', 'страна-изготовитель', 'единиц в одном товаре', 'тип'])

In [32]:
(
    df_matches
    .group_by("target")
    .agg(pl.count().alias("count"))
    .with_columns(
        (pl.col("count") / pl.sum("count")).alias("ratio")
    )
    .sort("target")
)

C:\Users\Samoylov_Nikita\AppData\Local\Temp\ipykernel_19472\3475110605.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("count"))


target,count,ratio
f64,u32,f64
0.0,271764,0.743227
1.0,93890,0.256773


## 5. Разбор датасета товаров

In [33]:
df_human.head()

id,name,attributes,category,normalized_attributes
i64,str,str,str,str
197,"""victor reinz прокладка впускного коллектора арт. 703315700""","""{""артикул"":""703315700"",""бренд"":""victor reinz"",""партномер (артикул производителя)"":""703315700"",""тип"":""прокладка двигателя"",""вид техники"":""легковые автомобили""}""","""Автотовары""","""{""артикул"": ""703315700"", ""бренд"": ""victor reinz"", ""партномер (артикул производителя)"": ""703315700"", ""тип"": ""прокладка двигателя"", ""тип системы"": ""легковые автомобили""}"""
415,"""stellox диск тормозной, арт. 8500892sx""","""{""артикул"":""stellox_8500892sx"",""место установки"":""передние"",""толщина тормозного диска, мм"":""28"",""бренд"":""stellox"",""oem-номер"":""3050086; a9064210012; 2e0615301; 68006716aa; 9064210012; 9064210212; 9064210012s; 9064210112"",""партномер (артикул производителя)"":""8500892sx"",""тип"":""диск тормозной"",""тип тормозного диска"":""вентилируемый"",""вид техники"":""легковые автомобили""}""","""Автотовары""","""{""артикул"": ""stellox_8500892sx"", ""система установки"": ""передние"", ""толщина тормозного диска, мм"": ""28"", ""бренд"": ""stellox"", ""oem-номер"": ""3050086; a9064210012; 2e0615301; 68006716aa; 9064210012; 9064210212; 9064210012s; 9064210112"", ""партномер (артикул производителя)"": ""8500892sx"", ""тип"": ""диск тормозной"", ""тип тормозного диска"": ""вентилируемый"", ""тип системы"": ""легковые автомобили""}"""
427,"""комплект подшипника ступицы колеса lynxauto""","""{""артикул производителя"":"""",""бренд"":""lynxauto"",""примечание"":""внешний вид изделия может отличаться от фотографий"",""цена за"":""1 шт."",""код товара"":""600000256748""}""","""Автотовары""","""{""артикул производителя"": """", ""бренд"": ""lynxauto"", ""примечание"": ""внешний вид изделия может отличаться от фотографий"", ""плата за"": ""1 шт."", ""код товара"": ""600000256748""}"""
1027,"""kraft подшипник ступицы, арт. kt204632, 1 шт.""","""{""артикул"":""112097-01"",""бренд"":""kraft"",""количество в упаковке, шт"":""1"",""партномер (артикул производителя)"":""kt204632"",""тип"":""подшипник ступицы""}""","""Автотовары""","""{""артикул"": ""112097-01"", ""бренд"": ""kraft"", ""количество в упаковке, шт"": ""1"", ""партномер (артикул производителя)"": ""kt204632"", ""тип"": ""подшипник ступицы""}"""
2639,"""фильтр салонный skoda fabia 00-""","""{""альтернативные артикулы товара"":""8104400xkz96a;50013702;ac0110c;50013941;9.7.84;la120;if-3019;afc1076;ac9403;ca-49010;6q0820367b;fs089;lak120;sab 123;1987432057;1010-026;6q0819653;6q0819653b;9.7.147;ac0110c;fs-089;dfc2545;if3019k;if3019p;sa1123;afw1076;gb9892c;fcr21f061;jdacx055;lac-1006;7110221sx;7110247sx;1cf031;nfe-2389;k1079;cuk2545;wc4015;7110543sx;st-6q0820367b;nf6123c;sa 1123;pf2123;mc-e4072;nf6123;1cf041;pf2041;lac1006c;8104400xkz96a\n"",""артикул"":""cu2545"",""бренд"":""sat"",""страна-изготовитель"":""китай"",""oem-номер"":""8104400xkz96a;50013702;ac0110c;50013941;9.7.84;la120;if-3019;afc1076;ac9403;ca-49010;6q0820367b;fs089;lak120;sab 123;1987432057;1010-026;6q0819653;6q0819653b;9.7.147;ac0110c;fs-089;dfc2545;if3019k;if3019p;sa1123;afw1076;gb9892c;fcr21f061;jdacx055;lac-1006;7110221sx;7110247sx;1cf031;nfe-2389;k1079;cuk2545;wc4015;7110543sx;st-6q0820367b;nf6123c;sa 1123;pf2123;mc-e4072;nf6123;1cf041;pf2041;lac1006c;8104400xkz96a\n"",""комплектация"":""фильтр салона для авто 1 шт"",""партномер (…","""Автотовары""","""{""альтернативные артикулы товара"": ""8104400xkz96a;50013702;ac0110c;50013941;9.7.84;la120;if-3019;afc1076;ac9403;ca-49010;6q0820367b;fs089;lak120;sab 123;1987432057;1010-026;6q0819653;6q0819653b;9.7.147;ac0110c;fs-089;dfc2545;if3019k;if3019p;sa1123;afw1076;gb9892c;fcr21f061;jdacx055;lac-1006;7110221sx;7110247sx;1cf031;nfe-2389;k1079;cuk2545;wc4015;7110543sx;st-6q0820367b;nf6123c;sa 1123;pf2123;mc-e4072;nf6123;1cf041;pf2041;lac1006c;8104400xkz96a"", ""артикул"": ""cu2545"", ""бренд"": ""sat"", ""страна-производитель"": ""китай"", ""oem-номер"": ""8104400xkz96a;50013702;ac0110c;500

In [34]:
df_human.sample().head()

id,name,attributes,category,normalized_attributes
i64,str,str,str,str
180388640587,"""кеды tendance""","""{""целевая аудитория"":""взрослая"",""бренд в одежде и обуви"":""tendance"",""страна бренда"":""франция"",""коллекция"":""базовая коллекция"",""цвет товара"":""темно-бежевый"",""страна-изготовитель"":""китай"",""материал подкладки обуви"":""натуральная кожа"",""материал верха"":""натуральная кожа"",""материал"":""натуральная кожа"",""российский размер"":""39"",""сезон"":""демисезон"",""пол"":""женский"",""высота подошвы, см"":""2.5"",""материал подошвы обуви"":""резина""}""","""Обувь""","""{""целевая аудитория"": ""взрослая"", ""бренд в одежде и обуви"": ""tendance"", ""страна бренда"": ""франция"", ""коллекция"": ""базовая коллекция"", ""цвет товара"": ""темно-бежевый"", ""страна-производитель"": ""китай"", ""материал подкладки обуви"": ""натуральная кожа"", ""материал высота"": ""натуральная кожа"", ""материал"": ""натуральная кожа"", ""российский количество"": ""39"", ""сезон"": ""демисезон"", ""пол"": ""женский"", ""высота подошвы, см"": ""2.5"", ""материал подошвы обуви"": ""резина""}"""


In [35]:
df_human = pd.read_parquet(items_human_path)
df_human.head()

,id,name,attributes,category,normalized_attributes
0,197,victor reinz прокладка впускного коллектора ар...,"{""артикул"":""703315700"",""бренд"":""victor reinz"",...",Автотовары,"{""артикул"": ""703315700"", ""бренд"": ""victor rein..."
1,415,"stellox диск тормозной, арт. 8500892sx","{""артикул"":""stellox_8500892sx"",""место установк...",Автотовары,"{""артикул"": ""stellox_8500892sx"", ""система уста..."
2,427,комплект подшипника ступицы колеса lynxauto,"{""артикул производителя"":"""",""бренд"":""lynxauto""...",Автотовары,"{""артикул производителя"": """", ""бренд"": ""lynxau..."
3,1027,"kraft подшипник ступицы, арт. kt204632, 1 шт.","{""артикул"":""112097-01"",""бренд"":""kraft"",""количе...",Автотовары,"{""артикул"": ""112097-01"", ""бренд"": ""kraft"", ""ко..."
4,2639,фильтр салонный skoda fabia 00-,"{""альтернативные артикулы товара"":""8104400xkz9...",Автотовары,"{""альтернативные артикулы товара"": ""8104400xkz..."


In [36]:
for index, row in df_human.head().iterrows():
    # Process each row
    attrs = json.loads(row['attributes'])
    print(attrs)
    attr_text = " ".join([f"{k}: {v}" for k, v in attrs.items()])
    print(attr_text)

{'артикул': '703315700', 'бренд': 'victor reinz', 'партномер (артикул производителя)': '703315700', 'тип': 'прокладка двигателя', 'вид техники': 'легковые автомобили'}
артикул: 703315700 бренд: victor reinz партномер (артикул производителя): 703315700 тип: прокладка двигателя вид техники: легковые автомобили
{'артикул': 'stellox_8500892sx', 'место установки': 'передние', 'толщина тормозного диска, мм': '28', 'бренд': 'stellox', 'oem-номер': '3050086; a9064210012; 2e0615301; 68006716aa; 9064210012; 9064210212; 9064210012s; 9064210112', 'партномер (артикул производителя)': '8500892sx', 'тип': 'диск тормозной', 'тип тормозного диска': 'вентилируемый', 'вид техники': 'легковые автомобили'}
артикул: stellox_8500892sx место установки: передние толщина тормозного диска, мм: 28 бренд: stellox oem-номер: 3050086; a9064210012; 2e0615301; 68006716aa; 9064210012; 9064210212; 9064210012s; 9064210112 партномер (артикул производителя): 8500892sx тип: диск тормозной тип тормозного диска: вентилируемый